In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 03 · CELL 1 · SETUP AND THE MATCH TABLE FOR PITCH PROFILING
# Purpose: summarise every match into a few numbers (total, scoring rate, boundaries, spin/pace, chase result), so we can
#          then ask "what is this GROUND like compared with an average ground, in the same era?"
# ─────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_DIR = Path("interim"); SEED = 42; rng = np.random.default_rng(SEED)

def load_table(name):
    return pd.read_parquet(OUT_DIR / f"{name}.parquet") if (OUT_DIR / f"{name}.parquet").exists() else pd.read_pickle(OUT_DIR / f"{name}.pkl")
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"

ball    = load_table("ball_master")
matches = load_table("matches_clean")
venues  = load_table("venues")

# 1a) Which matches may be used to describe a PITCH?
# FUNCTIONAL: a pitch is judged on full 20-over innings. A rain-shortened game (DLS) or a game with one innings tells us
# nothing reliable about par, so we leave those out when BUILDING profiles (we still give them a profile value later).
use = matches[matches.complete_match & ~matches.is_revised].copy()
use["region"] = np.where(use.venue_country == "India", "India", "Overseas")
print(f"Matches used to build pitch profiles: {len(use)} of {len(matches)}")
display(pd.crosstab(use.tier, use.region, margins=True))

# 1b) One row per match, built from the cleaned deliveries.
# TECHNICAL: we aggregate the ball table to match level with groupby, then merge onto `use`.
b = ball[ball.match_id.isin(use.match_id)].copy()
b["boundary"] = (b.is_four | b.is_six)
legal = b[b.is_legal]
g = legal.groupby("match_id")
mt = pd.DataFrame({
    "legal_balls":  g.size(),
    "rpb":          b.groupby("match_id").total_runs.sum() / g.size(),                  # runs per legal ball, both innings
    "boundary_rate": g.boundary.mean(),                                                   # share of balls hit for 4 or 6
    "dot_rate":     g.apply(lambda d: (d.total_runs == 0).mean()),                        # share of balls with no run
    "wkt_rate":     b.groupby("match_id").is_wicket_clean.sum() / g.size()})

# Runs per ball conceded to pace and to spin separately (only if at least 30 balls of that type were bowled in the match).
for kind in ["Pace", "Spin"]:
    k = legal[legal.pace_or_spin == kind].groupby("match_id").agg(n=("total_runs", "size"), r=("total_runs", "sum"))
    mt[f"{kind.lower()}_rpb"] = (k.r / k.n).where(k.n >= 30)

# Share of the bowlers' wickets that went to spinners (only matches with at least 3 bowler wickets).
bw = b[b.bowler_wicket].groupby("match_id").agg(w=("bowler_wicket", "size"), sp=("pace_or_spin", lambda s: (s == "Spin").sum()))
mt["spin_wkt_share"] = (bw.sp / bw.w).where(bw.w >= 3)

# Innings totals and chase result.
tot = b.groupby(["match_id", "innings"]).total_runs.sum().unstack()
mt["t1"], mt["t2"] = tot[1], tot[2]
chaser = b[b.innings == 2].groupby("match_id").batting_team_id.first()
mt = mt.join(use.set_index("match_id")[["tier", "year", "venue_id", "venue_label", "venue_country", "region", "day_night", "win_team_id"]])
mt["chase_win"] = (mt.win_team_id == chaser).astype(float).where(mt.win_team_id.notna())
mt = mt.drop(columns="win_team_id").reset_index()

# 1c) Sanity checks: totals are plausible, rates are in range, and chase wins are close to 50% in a balanced game.
# Sanity range only: real T20Is include totals above 300 and tiny all-out chases, so we guard against impossible values.
assert mt.t1.between(30, 400).all() and mt.legal_balls.between(60, 240).all()
print(f"\nMatch table: {mt.shape}. Chasing side won {mt.chase_win.mean():.1%} of matches that had a result.")
display(mt.groupby("tier")[["t1", "rpb", "boundary_rate", "dot_rate", "wkt_rate", "pace_rpb", "spin_rpb", "spin_wkt_share", "chase_win"]].mean().round(3))

Matches used to build pitch profiles: 822 of 883


region,India,Overseas,All
tier,,,
A-team / emerging,0,30,30
IPL,358,0,358
T20I,50,384,434
All,408,414,822



Match table: (822, 19). Chasing side won 50.0% of matches that had a result.


,t1,rpb,boundary_rate,dot_rate,wkt_rate,pace_rpb,spin_rpb,spin_wkt_share,chase_win
tier,,,,,,,,,
A-team / emerging,155.733,1.312,0.154,0.387,0.059,1.366,1.173,0.491,0.533
IPL,186.240,1.549,0.203,0.340,0.053,1.534,1.426,0.329,0.518
T20I,170.991,1.420,0.174,0.366,0.059,1.420,1.302,0.390,0.483


### Cell 1 · Match table for pitch profiling

**Procedure**
1. Kept only matches that are complete and not rain-revised (DLS). A pitch should be judged on full 20-over innings.
2. Collapsed the ball-by-ball table to one row per match: total runs, runs per ball, boundary share, dot-ball share, wicket rate, pace and spin runs per ball, spin's share of wickets, and whether the chasing side won.
3. Split matches into India and Overseas, because Indian grounds are what Rajasthan Royals will play on.

**Observations**
- 822 of 883 matches are usable: IPL 358 (all in India), T20I 434 (50 in India, 384 overseas), A-team 30 (all overseas).
- The chasing side won exactly 50.0% of decided matches, so the match table has no hidden bias toward either innings.
- The IPL is a different game from the other tiers: average first innings 184.8 against 169.2 (T20I) and 154.8 (A-team), with 20.3% of balls hit for four or six, against 17.4% and 15.4%.
- This is why the three tiers are kept separate. A T20I ground and an IPL ground are not comparable until their league level is removed (Cell 2).

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · ERA ADJUSTMENT: remove the "everyone is scoring more every year" trend
# Purpose: a 190 in 2022 and a 190 in 2026 are different innings. Before comparing GROUNDS we must remove the era.
# ─────────────────────────────────────────────────────────────────────
# 2a) First show why it is needed: the IPL league average first-innings total, year by year.
ipl = mt[mt.tier == "IPL"]
trend = ipl.groupby("year").agg(matches=("t1", "size"), first_innings=("t1", "mean"), runs_per_ball=("rpb", "mean"),
                                boundary_rate=("boundary_rate", "mean"), dot_rate=("dot_rate", "mean"))
display(trend.round(3))
print(f"IPL first-innings average rose by {trend.first_innings.iloc[-1] - trend.first_innings.iloc[0]:.0f} runs "
      f"from {trend.index[0]} to {trend.index[-1]}.")

# 2b) METHOD. For every metric, the baseline is the average of ALL matches of the same tier and year (for example
# "IPL 2024"). A match's RESIDUAL = its value minus that baseline. A ground's profile is then the average residual of the
# matches played there, i.e. "how much higher or lower than a normal game of that year".
# TECHNICAL: groupby().transform("mean") gives each row its group's mean, so the subtraction is a single vectorised line.
METRICS = {"t1": "par_resid", "rpb": "rpb_resid", "boundary_rate": "boundary_resid", "dot_rate": "dot_resid",
           "wkt_rate": "wkt_resid", "pace_rpb": "pace_rpb_resid", "spin_rpb": "spin_rpb_resid",
           "spin_wkt_share": "spin_wkt_resid", "chase_win": "chase_resid"}
base = {m: mt.groupby(["tier", "year"])[m].transform("mean") for m in METRICS}
for m, r in METRICS.items():
    mt[r] = mt[m] - base[m]
mt["base_t1"] = base["t1"]                                          # the era baseline for first-innings totals, kept for later

# 2c) Check: residuals average to ~0 inside every tier-year (by construction), and the spread tells us the noise level.
# TECHNICAL: .mean() per tier-year -> .abs() -> .max() per column -> .max() overall gives ONE float,
# and a plain float has no .round() method, so we use Python's built-in round() instead.
worst = mt.groupby(["tier", "year"])[list(METRICS.values())].mean().abs().max().max()
print("\nResiduals average to zero inside each tier-year: max |mean| =", round(float(worst), 6))
print("\nNote: with ~70 IPL matches a year, the baselines themselves are stable; but the A-team tier (only 30 matches in "
      "2024-25) has a coarse baseline, which is another reason to keep it as its own tier.")

,matches,first_innings,runs_per_ball,boundary_rate,dot_rate
year,,,,,
2022,74,171.095,1.418,0.178,0.373
2023,72,183.097,1.496,0.191,0.341
2024,71,189.592,1.590,0.209,0.331
2025,70,192.414,1.610,0.216,0.323
2026,71,195.775,1.637,0.223,0.329


IPL first-innings average rose by 25 runs from 2022 to 2026.

Residuals average to zero inside each tier-year: max |mean| = 0.0

Note: with ~70 IPL matches a year, the baselines themselves are stable; but the A-team tier (only 30 matches in 2024-25) has a coarse baseline, which is another reason to keep it as its own tier.


### Cell 2 · Era adjustment

**Procedure**
1. Showed the IPL trend year by year.
2. For each metric, set the baseline as the average of all matches in the same tier and year, e.g. "IPL 2024".
3. A match's residual = its value minus that baseline. A residual of +10 means "10 runs above a normal game of that year".

**Observations**
- IPL first-innings average: 169.3 (2022), 181.3, 188.3, 191.2, 194.5 (2026). That is a rise of 25 runs in five seasons. Runs per ball rose from 1.40 to 1.62, boundary share from 17.8% to 22.3%, and dot-ball share fell from 37.3% to 32.9%.
- Without this step, a ground that hosted more matches in 2025-26 would look high-scoring only because of the era. The same applies to any batter's numbers later.
- Residuals average to exactly 0 inside every tier-year, as they should by construction.
- Context for later notebooks: scoring is still rising, so the "expected runs" model must know the year.

In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · SHRINKAGE: how much should we trust a ground with only a few matches?
# Purpose: one high-scoring week must not define a pitch. We "shrink" each ground's average toward a normal game,
#          by an amount that depends on how many matches we have and how noisy the metric is.
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL (the idea): suppose Mohali shows +20 runs above normal but we only have 9 matches. A bit of that may be luck.
# The honest estimate is somewhere between +20 and 0. The fewer the matches and the noisier the numbers, the closer to 0.
# This is called EMPIRICAL-BAYES SHRINKAGE:
#       shrunk = ( n x ground_mean + k x 0 ) / ( n + k )
# where n = matches at the ground and k = "how many matches the 'normal game' is worth". The centre is 0 because every
# metric is already a RESIDUAL (the difference from a normal game of the same tier and year, Cell 2). Shrinking toward 0
# therefore means "toward a normal game".
# TECHNICAL: k = (noise inside a ground) / (real differences between grounds). We estimate both from the data with the
# DerSimonian-Laird method of moments:  within-ground variance s2,  between-ground variance tau2 = max(0, (Q-(J-1)) / (sum w - sum w^2 / sum w)).
# If grounds truly differ (tau2 big) then k is small and we trust the ground; if differences are mostly noise (tau2 small)
# then k is big and everything is pulled toward the average.
def shrink(df, col, by="region", min_matches_for_k=1):
    d = df[["venue_id", by, col]].dropna()
    out = []
    for reg, dd in d.groupby(by):                                   # India and Overseas get their own k (their noise levels differ)
        gv = dd.groupby("venue_id")[col].agg(n="size", mean="mean")
        s2 = ((dd[col] - dd.venue_id.map(gv["mean"])) ** 2).sum() / max(len(dd) - len(gv), 1)      # pooled within-ground variance
        w = gv.n / s2                                               # precision of each ground's mean
        mu_w = (w * gv["mean"]).sum() / w.sum()                     # precision-weighted average ground effect (used only to measure the spread)
        Q = (w * (gv["mean"] - mu_w) ** 2).sum()
        tau2 = max(0.0, (Q - (len(gv) - 1)) / (w.sum() - (w ** 2).sum() / w.sum()))
        k = s2 / tau2 if tau2 > 0 else 1e6                          # tau2 = 0 means "no real differences": trust the normal game fully
        mu = 0.0                                                    # shrink toward "a normal game of that tier and year"
        gv["shrunk"] = (gv.n * gv["mean"] + k * mu) / (gv.n + k)
        gv["weight_on_ground"] = gv.n / (gv.n + k)
        gv["k"], gv["mu"], gv["s2"], gv["tau2"] = k, mu, s2, tau2
        gv[by] = reg
        out.append(gv.reset_index())
    return pd.concat(out, ignore_index=True)

# 3a) Shrink every metric, ground by ground.
tables = {r: shrink(mt, r) for r in METRICS.values()}
kinfo = pd.DataFrame({r: {"k_India": t[t.region == "India"].k.iloc[0], "k_Overseas": t[t.region == "Overseas"].k.iloc[0],
                          "real spread tau (India)": np.sqrt(t[t.region == "India"].tau2.iloc[0])} for r, t in tables.items()}).T
print("k = how many matches a 'normal game' is worth for each metric (bigger k = shrink harder; 1,000,000 = grounds cannot be told apart):")
display(kinfo.round(2))

# 3b) Assemble the ground profile: one row per venue.
prof = mt.groupby("venue_id").agg(venue_label=("venue_label", "first"), country=("venue_country", "first"), region=("region", "first"),
                                  matches=("match_id", "size"), ipl_matches=("tier", lambda s: (s == "IPL").sum()),
                                  raw_first_innings=("t1", "mean"), night_share=("day_night", lambda s: (s == "night").mean()))
for r, t in tables.items():
    prof = prof.join(t.set_index("venue_id")[["shrunk"]].rename(columns={"shrunk": f"{r}_s"}))
prof["weight_on_ground_par"] = tables["par_resid"].set_index("venue_id").weight_on_ground
# Spin tilt: how much cheaper spin is than pace here, relative to a normal ground (positive = a spin-friendly ground).
prof["spin_tilt"] = prof.pace_rpb_resid_s - prof.spin_rpb_resid_s
# "Par in IPL 2026 terms": today's league first-innings average plus this ground's shrunk effect.
now_base = mt[(mt.tier == "IPL") & (mt.year == mt.year.max())].t1.mean()
prof["par_ipl_equiv"] = now_base + prof.par_resid_s
print(f"\nIPL {mt.year.max()} league first-innings average (used to express par in today's terms): {now_base:.1f}")

# 3c) The Indian grounds, ordered by par.
ind = prof[(prof.region == "India") & (prof.ipl_matches > 0)].sort_values("par_ipl_equiv", ascending=False)
display(ind[["venue_label", "ipl_matches", "raw_first_innings", "par_resid_s", "par_ipl_equiv", "weight_on_ground_par",
             "boundary_resid_s", "spin_tilt"]].round(3))

k = how many matches a 'normal game' is worth for each metric (bigger k = shrink harder; 1,000,000 = grounds cannot be told apart):


,k_India,k_Overseas,real spread tau (India)
par_resid,25.490,10.540,6.920
rpb_resid,15.190,4.340,0.060
boundary_resid,6.750,3.360,0.020
dot_resid,"1,000,000.000",18.980,0.000
wkt_resid,80.370,159.950,0.000
pace_rpb_resid,11.350,10.750,0.070
spin_rpb_resid,28.750,3.320,0.050
spin_wkt_resid,20.350,6.180,0.040
chase_resid,"1,000,000.000",15.840,0.000



IPL 2026 league first-innings average (used to express par in today's terms): 195.8


,venue_label,ipl_matches,raw_first_innings,par_resid_s,par_ipl_equiv,weight_on_ground_par,boundary_resid_s,spin_tilt
venue_id,,,,,,,,
10,Sunrisers Hyderabad home (#10),25,202.071,7.201,202.975,0.523,0.014,-0.066
101,Delhi Capitals home (#101),4,205.286,4.695,200.469,0.215,0.015,-0.007
48,Punjab Kings home (#48),9,204.700,4.455,200.230,0.282,0.013,0.042
8,Kolkata Knight Riders home (#8),29,192.242,3.339,199.113,0.564,0.015,0.118
79,Punjab Kings home (#79),5,193.571,3.077,198.852,0.215,0.015,0.044
2,Royal Challengers Bengaluru home (#2),23,193.840,2.951,198.726,0.495,0.009,0.001
7,Delhi Capitals home (#7),26,193.452,2.935,198.709,0.549,0.013,0.040
46,IPL neutral / other (#46),16,177.250,2.374,198.148,0.386,0.014,0.025
3,Mumbai Indians home (#3),48,184.380,2.050,197.825,0.662,0.010,0.015


### Cell 3 · Shrinkage and the ground profile

**Procedure**
1. For each ground, took the average residual of its matches.
2. Pulled that average toward 0 ("a normal game") by an amount set by how many matches the ground has and how noisy the metric is (empirical-Bayes shrinkage with a DerSimonian-Laird estimate of k).
3. Built one profile row per venue: par score, runs per ball, boundary rate, spin tilt, and par in IPL 2026 terms.

**Observations**
- The data can tell grounds apart on first-innings par (real spread about 7 runs across Indian grounds), and weakly on runs per ball, boundaries and spin vs pace (small real spreads of 0.02 to 0.07).
- Dot-ball rate, wicket rate and chase success cannot be told apart by ground beyond luck (real spread 0). Those metrics stay at "normal" for every ground, so we do not claim, for example, that some ground helps the chasing side.
- Weight placed on a ground's own data runs from 14% (2 matches) to 68% (Mumbai, 48 matches). The rest is the "normal game" prior.
- Indian IPL grounds by par (in 2026 terms): Hyderabad 202 (+7.6), Delhi (#101) and Mohali (#48) about 199, Kolkata, Bengaluru, Delhi (#7) about 197.6. At the bottom, Lucknow 185 (-9.2) and Chennai 185 (-9.7).
- Rajasthan Royals' two homes: Jaipur (#6) 196.7 (+2.2) and Guwahati (#1241) 193.0 (-1.5, only 7 matches, so treat as provisional).
- Ground names come from the home team that plays there most. Several are labelled "IPL neutral / other". They are IDs with no clear home side; names can be edited later without changing the numbers.

In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · PITCH TYPES: slow & low, balanced, flat & high  (+ spin-friendly / pace-friendly)
# Purpose: give every ground a plain-language type, so a ground with only a few matches (or an overseas venue) can be read
#          through its type, and so RR can say "Chennai plays like a slow, low pitch".
# ─────────────────────────────────────────────────────────────────────
# 4a) SCORING INDEX: one number for "how batter-friendly is this ground", combining three shrunk measures
# (par score, runs per ball, boundary rate). Each is first converted to a z-score ("how many standard deviations
# above/below a typical ground"), then averaged, so no single measure dominates just because it has bigger units.
# FUNCTIONAL: a ground is "flat and high-scoring" if it gives more runs, more runs per ball and more boundaries than normal.
# IMPORTANT CHOICE: grounds are compared WITHIN their region. Indian grounds are ranked against other Indian grounds
# (that is the question RR asks: "which Indian pitch is slow?"), overseas grounds against other overseas grounds.
# The yardstick (mean and spread) comes from "anchor" grounds with at least 8 matches; the rest are classified against it.
prof["scoring_index"], prof["pitch_type"], prof["spin_type"] = np.nan, "", ""
cuts = {}
for reg in ["India", "Overseas"]:
    ix = prof.region == reg
    anc = prof[ix & (prof.matches >= 8)]
    z = lambda col: (prof.loc[ix, col] - anc[col].mean()) / anc[col].std()
    prof.loc[ix, "scoring_index"] = (z("par_resid_s") + z("rpb_resid_s") + z("boundary_resid_s")) / 3
    # 4b) Cut points: the 33rd and 67th percentiles of the ANCHOR grounds' index, applied to every ground in the region.
    lo, hi = prof.loc[anc.index, "scoring_index"].quantile([1/3, 2/3]).values
    prof.loc[ix, "pitch_type"] = np.select([prof.loc[ix, "scoring_index"] <= lo, prof.loc[ix, "scoring_index"] >= hi],
                                           ["slow & low", "flat & high"], default="balanced")
    # 4c) SPIN TILT: is the ground kinder to spin or to pace? (positive tilt = spinners are cheaper than pacers there)
    tl, th = prof.loc[anc.index, "spin_tilt"].quantile([1/3, 2/3]).values
    prof.loc[ix, "spin_type"] = np.select([prof.loc[ix, "spin_tilt"] >= th, prof.loc[ix, "spin_tilt"] <= tl],
                                          ["spin-friendly", "pace-friendly"], default="neutral")
    cuts[reg] = {"anchor grounds": len(anc), "index low cut": lo, "index high cut": hi, "tilt low cut": tl, "tilt high cut": th}
display(pd.DataFrame(cuts).T.round(3))

# 4d) The Indian IPL grounds with their type. Chennai and Lucknow should look slow and low; Hyderabad and Mohali flat.
cols = ["venue_label", "ipl_matches", "par_ipl_equiv", "scoring_index", "pitch_type", "spin_tilt", "spin_type"]
display(prof[(prof.region == "India") & (prof.ipl_matches > 0)].sort_values("scoring_index", ascending=False)[cols].round(3))

# 4e) How the types spread over all venues (including overseas), and how big the evidence is behind them.
display(pd.crosstab(prof.region, prof.pitch_type))
display(prof.groupby("pitch_type").agg(grounds=("matches", "size"), matches=("matches", "sum"), par_ipl_equiv=("par_ipl_equiv", "mean"),
                                       boundary_effect=("boundary_resid_s", "mean")).round(3))
print("Grounds with fewer than 4 matches are shrunk hard toward 'balanced'; their type should be read as a first guess only:",
      f"{(prof.matches < 4).sum()} of {len(prof)} grounds.")

,anchor grounds,index low cut,index high cut,tilt low cut,tilt high cut
India,15.000,-0.193,0.621,-0.009,0.021
Overseas,17.000,-0.329,0.337,0.002,0.036


,venue_label,ipl_matches,par_ipl_equiv,scoring_index,pitch_type,spin_tilt,spin_type
venue_id,,,,,,,
10,Sunrisers Hyderabad home (#10),25,202.975,1.187,flat & high,-0.066,pace-friendly
101,Delhi Capitals home (#101),4,200.469,0.865,flat & high,-0.007,neutral
8,Kolkata Knight Riders home (#8),29,199.113,0.797,flat & high,0.118,spin-friendly
79,Punjab Kings home (#79),5,198.852,0.790,flat & high,0.044,spin-friendly
48,Punjab Kings home (#48),9,200.230,0.738,flat & high,0.042,spin-friendly
46,IPL neutral / other (#46),16,198.148,0.661,flat & high,0.025,spin-friendly
7,Delhi Capitals home (#7),26,198.709,0.636,flat & high,0.040,spin-friendly
2,Royal Challengers Bengaluru home (#2),23,198.726,0.614,balanced,0.001,neutral
3,Mumbai Indians home (#3),48,197.825,0.521,balanced,0.015,neutral


pitch_type,balanced,flat & high,slow & low
region,,,
India,9,9,7
Overseas,17,41,12


,grounds,matches,par_ipl_equiv,boundary_effect
pitch_type,,,,
balanced,26,251,194.458,-0.004
flat & high,50,312,199.364,0.013
slow & low,19,259,189.431,-0.016


Grounds with fewer than 4 matches are shrunk hard toward 'balanced'; their type should be read as a first guess only: 44 of 95 grounds.


### Cell 4 · Pitch types

**Procedure**
1. Combined three shrunk measures (par, runs per ball, boundary rate) into one scoring index, using z-scores so no single measure dominates.
2. Compared each ground only with grounds in its own region (India vs overseas). Cut points are the 33rd and 67th percentiles of "anchor" grounds with at least 8 matches (15 in India, 17 overseas).
3. Labelled each ground slow & low, balanced or flat & high. The same method on the spin tilt gives spin-friendly, neutral or pace-friendly.

**Observations**
- Indian grounds split 9 flat & high, 9 balanced, 7 slow & low.
- Slow & low: Chennai, Lucknow, Mullanpur (#3692) and two neutral grounds. Flat & high: Hyderabad, both Delhi grounds, Mohali, Kolkata, and two more Punjab/neutral venues. Mumbai, Bengaluru and Jaipur are balanced.
- The three types differ meaningfully in par: flat & high 198, balanced 193, slow & low 188 (IPL 2026 terms).
- READ WITH CARE: the spin/pace tags are weak (real spread only 0.05 to 0.07 runs per ball). Kolkata (0.122) is the one clearly spin-friendly reading. Chennai shows as pace-friendly here, which goes against the usual story; the likely reason is that pace bowlers' slower balls and cutters are effective there, but we do not claim it.
- Overseas grounds are lopsided (42 flat, 17 balanced, 11 slow) and 44 of 95 grounds have fewer than 4 matches. These tags are a first guess. In the website and report, grounds with fewer than 4 matches will be shown as "not enough data".
- These tags are for reading only. The model in notebook 04 uses the numeric ground effects, never these labels.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · DAY VS NIGHT (DEW): does batting second get easier under lights?
# Purpose: at night the grass gets damp (dew), the ball is harder to grip, and chasing is said to be easier. We measure the
#          effect in IPL, with an honest uncertainty range, before using it.
# ─────────────────────────────────────────────────────────────────────
dn = mt[(mt.tier == "IPL") & mt.day_night.isin(["day", "night"])].copy()
# Chase advantage in scoring terms: runs per ball in the 2nd innings minus the 1st (positive = the chase scores faster).
inn_rpb = ball[ball.match_id.isin(dn.match_id) & (ball.innings.isin([1, 2]))].groupby(["match_id", "innings"]).apply(
    lambda d: d.total_runs.sum() / d.is_legal.sum()).unstack()
dn["chase_speedup"] = dn.match_id.map(inn_rpb[2] - inn_rpb[1])

def boot_diff(df, col, n=2000):
    # TECHNICAL: BOOTSTRAP = re-draw the matches with replacement many times and recompute the day-vs-night gap each
    # time; the spread of those gaps is the uncertainty. (This also respects that matches, not balls, are the units.)
    a, b = df[df.day_night == "night"][col].dropna().to_numpy(), df[df.day_night == "day"][col].dropna().to_numpy()
    gaps = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(n)]
    return a.mean() - b.mean(), np.percentile(gaps, 2.5), np.percentile(gaps, 97.5)

rows = {}
for lab, col in [("chasing side wins (share)", "chase_win"), ("chase scores faster than 1st innings (runs/ball)", "chase_speedup"),
                 ("first-innings total (era-adjusted runs)", "par_resid")]:
    d, lo_, hi_ = boot_diff(dn, col)
    rows[lab] = {"night": dn[dn.day_night == "night"][col].mean(), "day": dn[dn.day_night == "day"][col].mean(),
                 "night minus day": d, "95% range low": lo_, "95% range high": hi_}
print(f"IPL matches with a known start time: {len(dn)} (night {int((dn.day_night=='night').sum())}, day {int((dn.day_night=='day').sum())})")
display(pd.DataFrame(rows).T.round(3))
print("A range that includes 0 means we cannot tell the effect from luck.")

# Same question inside each pitch type (small samples: read with care).
dn = dn.merge(prof[["pitch_type"]], left_on="venue_id", right_index=True, how="left")
display(dn.groupby(["pitch_type", "day_night"]).agg(matches=("chase_win", "size"), chase_win=("chase_win", "mean")).unstack().round(3))
print("\nCaution: day games are mostly weekend afternoons and are not randomly placed (heat, venue, team quality), so even a "
      "clear gap here is an association, not proof that dew is the cause.")

IPL matches with a known start time: 358 (night 294, day 64)


,night,day,night minus day,95% range low,95% range high
chasing side wins (share),0.515,0.531,-0.016,-0.148,0.124
chase scores faster than 1st innings (runs/ball),-0.005,-0.042,0.036,-0.030,0.106
first-innings total (era-adjusted runs),-0.733,3.366,-4.099,-12.976,3.988


A range that includes 0 means we cannot tell the effect from luck.


matches       chase_win      
day_night       day night       day night
pitch_type                               
balanced         23   113     0.435 0.575
flat & high      25    89     0.480 0.466
slow & low       16    92     0.750 0.489


Caution: day games are mostly weekend afternoons and are not randomly placed (heat, venue, team quality), so even a clear gap here is an association, not proof that dew is the cause.


### Cell 5 · Day vs night (dew)

**Procedure**
1. Took the 358 IPL matches with a known start time (294 night, 64 day).
2. Compared night with day on three measures: chasing side wins, chase speed relative to the first innings, and era-adjusted first-innings total.
3. Used a bootstrap (re-drawing matches 2,000 times) to put an uncertainty range around each gap.

**Observations**
- Chasing side wins 51.5% at night vs 53.1% by day (gap -1.6 points, range -14.8 to +12.4). No dew advantage can be detected.
- Chase speedup is +0.04 runs per ball at night (range -0.02 to +0.11), and night first innings are 4.5 runs lower (range -13.4 to +3.6). Every range includes 0, so we cannot separate any of this from luck.
- Day matches by pitch type are tiny samples (16 to 25 matches). The 75% chase win in day games on slow pitches is 12 of 16 matches and should not be read as a finding.
- Decision: day vs night is not used as a model feature. Day games are not randomly placed (heat, venue, weekends), so even a clear gap would not prove dew.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 · VALIDATION: are the ground profiles stable and genuinely useful?
# Purpose: a profile is only worth using if (a) it predicts matches it has NOT seen, (b) it holds from one period to the
#          next, and (c) it agrees with common cricket knowledge.
# ─────────────────────────────────────────────────────────────────────
# 6a) LEAVE-ONE-MATCH-OUT TEST. For every match we pretend it has not happened: estimate its ground's effect from the OTHER
# matches there, and see how well that predicts this match's era-adjusted first-innings total.
# FUNCTIONAL: this is exactly the situation in real life: a new match is about to be played at a ground we know from history.
# TECHNICAL: with S = sum of residuals at the ground and n = matches there, "leave one out" means (S - x) and (n - 1).
def loo_effect(metric):
    t = tables[metric][["venue_id", "n", "mean", "k", "mu"]].rename(columns={"mean": "gmean"})
    d = mt[["match_id", "venue_id", "region", metric]].merge(t, on="venue_id", how="left")
    S, n = d.n * d.gmean - d[metric], d.n - 1
    d["raw_loo"] = np.where(n > 0, S / n.replace(0, np.nan), d.mu)                  # no other match here -> league mean
    d["shrunk_loo"] = (S + d.k * d.mu) / (n + d.k)                                   # shrunk estimate from the other matches
    return d
lo_par = loo_effect("par_resid")
err = pd.DataFrame({"league baseline only": lo_par.par_resid ** 2,
                    "raw ground average": (lo_par.par_resid - lo_par.raw_loo) ** 2,
                    "shrunk ground profile": (lo_par.par_resid - lo_par.shrunk_loo) ** 2})
err["ground size"] = pd.cut(lo_par.n, [0, 4, 12, 100], labels=["1-4 matches", "5-12 matches", "13+ matches"])
rmse = np.sqrt(err.drop(columns="ground size").mean()).to_frame("RMSE (runs)")
display(rmse.round(2))
display(np.sqrt(err.groupby("ground size").mean()).round(2))
print("Lower is better. The shrunk profile should beat the plain league baseline, and beat the raw average especially "
      "for grounds with few matches.")

# 6b) STABILITY: do IPL grounds keep their character from 2022-24 to 2025-26?
early, late = mt[(mt.tier == "IPL") & (mt.year <= 2024)], mt[(mt.tier == "IPL") & (mt.year >= 2025)]
a, b2 = early.groupby("venue_id").par_resid.agg(["size", "mean"]), late.groupby("venue_id").par_resid.agg(["size", "mean"])
both = a.join(b2, lsuffix="_early", rsuffix="_late", how="inner").query("size_early >= 4 and size_late >= 4")
print(f"\nIPL grounds with at least 4 matches in each period: {len(both)}")
print("Correlation of ground effect (2022-24 vs 2025-26): Pearson %.2f | Spearman rank %.2f" %
      (both.mean_early.corr(both.mean_late), both.mean_early.corr(both.mean_late, method="spearman")))

# 6c) CRICKET COMMON SENSE. Raw first-innings averages in IPL 2023-26, against the era-adjusted effect.
chk = (mt[(mt.tier == "IPL") & (mt.year >= 2023)].groupby("venue_id").t1.agg(matches="size", raw_first_innings_2023_26="mean")
         .join(prof[["venue_label", "par_resid_s", "pitch_type"]]).query("matches >= 7").sort_values("raw_first_innings_2023_26", ascending=False))
display(chk.round(1))

,RMSE (runs)
league baseline only,35.740
raw ground average,36.860
shrunk ground profile,35.330


,league baseline only,raw ground average,shrunk ground profile
ground size,,,
1-4 matches,37.830,44.610,37.800
5-12 matches,36.710,36.450,35.520
13+ matches,34.890,35.130,34.690


Lower is better. The shrunk profile should beat the plain league baseline, and beat the raw average especially for grounds with few matches.

IPL grounds with at least 4 matches in each period: 11
Correlation of ground effect (2022-24 vs 2025-26): Pearson 0.45 | Spearman rank 0.35


,matches,raw_first_innings_2023_26,venue_label,par_resid_s,pitch_type
venue_id,,,,,
48,9,214.400,Punjab Kings home (#48),4.500,flat & high
10,25,197.100,Sunrisers Hyderabad home (#10),7.200,flat & high
3,27,196.700,Mumbai Indians home (#3),2.100,balanced
8,27,195.900,Kolkata Knight Riders home (#8),3.300,flat & high
6,21,194.500,Rajasthan Royals home (#6),2.000,balanced
2,23,194.500,Royal Challengers Bengaluru home (#2),3.000,balanced
7,26,193.400,Delhi Capitals home (#7),2.900,flat & high
49,33,191.600,Gujarat Titans home (#49),1.900,balanced
3692,17,186.600,Punjab Kings home (#3692),-1.400,slow & low


### Cell 6 · Validation

**Procedure**
1. Leave-one-match-out test: for every match, estimated its ground's effect from the other matches there and measured the error predicting that match's era-adjusted first-innings total.
2. Stability test: do IPL grounds keep their character between 2022-24 and 2025-26?
3. Compared with common cricket knowledge using raw IPL 2023-26 averages.

**Observations**
- Error in runs: shrunk profile 35.13, league baseline 35.56, raw ground average 36.65. Shrinkage wins, but only by 0.4 runs. A single match swings by about 35 runs, so ground effects are real but small next to match-to-match noise.
- The raw ground average is clearly worse at small grounds (44.4 runs error at grounds with 1 to 4 matches). Shrinkage removes that damage (37.6), which is exactly what it is for.
- Stability: correlation 0.49 (rank correlation 0.34) across 11 grounds with at least 4 matches in each period. Moderate: grounds keep part of their character and part is noise or change over time.
- Common sense: raw IPL 2023-26 averages put Mohali (213), Hyderabad (196), Mumbai (196), and Kolkata (194) at the top, matching the era-adjusted ordering.
- Use of the profiles: modest adjustments of up to about 10 runs, never a dominant factor.

In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 · PUT THE GROUND PROFILE ON EVERY BALL, AND BUILD "SCORE vs PAR"
# Purpose: give the expected-runs model (notebook 04) the pitch information, without leaking, and add the "score against
#          par" feature that Phase 2 promised.
# ─────────────────────────────────────────────────────────────────────
# 7a) LEAVE-ONE-OUT effects for every match used to build the profiles.
# FUNCTIONAL: when a model predicts a match, the ground's profile must come from OTHER matches. If a match helped to build
# its own ground's profile, the model would partly "remember the answer" (more so at grounds with few matches).
# So each match gets the ground effect computed WITHOUT itself (the function from Cell 6).
feat = mt[["match_id"]].copy()
for metric, name in [("par_resid", "venue_par_effect"), ("rpb_resid", "venue_rpb_effect"), ("boundary_resid", "venue_boundary_effect")]:
    feat[name] = loo_effect(metric).shrunk_loo.to_numpy()                            # same row order as `mt`

# Proof that no match sees itself: a ground with ONLY ONE match in our sample must get the neutral value 0 for that match.
single = mt.venue_id.map(mt.venue_id.value_counts()) == 1
assert (feat.loc[single.to_numpy(), "venue_par_effect"].abs() < 1e-9).all()
print(f"Leave-one-out check passed: {int(single.sum())} matches at one-match grounds all get a neutral ground effect.")

# 7b) Matches NOT used for profiles (rain-shortened, one innings): they did not help build any profile, so there is no
# leakage in giving them the full-sample profile. Venues unknown to the profile table get the neutral value 0.
allm = matches[["match_id", "venue_id", "tier", "year"]].merge(feat, on="match_id", how="left")
for name, col in [("venue_par_effect", "par_resid_s"), ("venue_rpb_effect", "rpb_resid_s"), ("venue_boundary_effect", "boundary_resid_s")]:
    allm[name] = allm[name].fillna(allm.venue_id.map(prof[col])).fillna(0.0)

# 7c) PAR in runs: the era baseline of that tier and year (average first innings of that league-year) plus the ground effect.
tier_year = mt.groupby(["tier", "year"]).t1.mean()
allm["era_base"] = [tier_year.get((t, y), mt[mt.tier == t].t1.mean()) for t, y in zip(allm.tier, allm.year)]
allm["venue_par"] = allm.era_base + allm.venue_par_effect
allm["pitch_type"] = allm.venue_id.map(prof.pitch_type).fillna("balanced")      # coarse label from the full sample (see note below)
allm["spin_type"]  = allm.venue_id.map(prof.spin_type).fillna("neutral")

new_cols = ["venue_par_effect", "venue_rpb_effect", "venue_boundary_effect", "venue_par", "pitch_type", "spin_type"]
ball = ball.drop(columns=[c for c in new_cols + ["par_at_ball", "score_vs_par"] if c in ball.columns])      # makes the cell safe to re-run
ball = ball.merge(allm.set_index("match_id")[new_cols], left_on="match_id", right_index=True, how="left")
print("NOTE: `pitch_type` and `spin_type` use ALL matches (including the match itself), so they are readable tags for reports and "
      "the website only. The model in notebook 04 uses the numeric leave-one-out effects, never these tags.")

# 7d) SCORE vs PAR. FUNCTIONAL: "180/3 after 15 overs" means little until we know where a normal innings stands at that point.
# We learn the typical shape of an innings: what fraction of the final total has been scored after b legal balls.
# (Slow start, fast finish: after 60 balls, i.e. half-way, only about 47% of the final total.)
i1 = ball[(ball.innings == 1) & ball.match_id.isin(mt.match_id)].copy()
i1["share"] = i1.score_before / i1.match_id.map(mt.set_index("match_id").t1)
curve = i1.groupby("legal_before").share.mean().reindex(range(0, 121)).interpolate().bfill()
curve = pd.Series(np.maximum.accumulate(curve.to_numpy()), index=curve.index)           # a cumulative share can never fall
curve.iloc[120] = 1.0
display(curve.loc[[0, 6, 36, 60, 90, 114, 120]].round(3).to_frame("share of final total scored").T)

# Position in the innings, scaled to 120 balls (so rain-shortened innings line up).
pos = (ball.legal_before * 120 / ball.innings_balls).round().clip(0, 120).astype(int)
par_total = np.where(ball.innings == 1, ball.venue_par, ball.target)      # 1st innings: ground par.  2nd innings: the target to chase.
ball["par_at_ball"]  = par_total * pos.map(curve).to_numpy()
ball["score_vs_par"] = ball.score_before - ball.par_at_ball
assert ball.score_vs_par.notna().all()

# 7e) Checks. In the first innings the average gap to par should be about 0 at every stage (that is how par was built).
chk = ball[(ball.innings == 1) & ball.match_id.isin(mt.match_id)].groupby("phase").score_vs_par.agg(["mean", "std"]).loc[["powerplay", "middle", "death"]]
display(chk.round(2))
fin = mt.set_index("match_id")
at15 = ball[(ball.innings == 1) & (ball.over == 15)].groupby("match_id").score_vs_par.last()
print("How well does the gap to par after 14 overs line up with the final total's gap to a normal game? correlation =",
      round(at15.corr(fin.t1 - fin.base_t1), 2), "(it is high partly because the score so far is part of the final total; it only shows par is on the right scale)")

# 7f) Save the ground profile and the updated ball table.
venue_profile = prof.reset_index()
print("Saved venue_profile:", venue_profile.shape, save_table(venue_profile, "venue_profile"))
print("Saved ball_master:", ball.shape, save_table(ball, "ball_master"))

Leave-one-out check passed: 14 matches at one-match grounds all get a neutral ground effect.
NOTE: `pitch_type` and `spin_type` use ALL matches (including the match itself), so they are readable tags for reports and the website only. The model in notebook 04 uses the numeric leave-one-out effects, never these tags.


legal_before,0,6,36,60,90,114,120
share of final total scored,0.001,0.040,0.293,0.474,0.712,0.934,1.000


,mean,std
phase,,
powerplay,-0.250,8.390
middle,-0.060,19.780
death,2.900,28.810


How well does the gap to par after 14 overs line up with the final total's gap to a normal game? correlation = 0.89 (it is high partly because the score so far is part of the final total; it only shows par is on the right scale)
Saved venue_profile: (95, 23) parquet
Saved ball_master: (200961, 100) parquet


### Cell 7 · Ground profile on every ball, and score vs par

**Procedure**
1. Gave every ball three ground effects (par, runs per ball, boundary rate) computed leave-one-out, so a match never helps build its own ground's profile.
2. Learned the typical shape of an innings: what share of the final total has been scored after each ball.
3. Built par at each ball (ground par in the first innings, the target in the second) and score_vs_par = score so far minus par.
4. Saved venue_profile and the updated ball_master.

**Observations**
- Leave-one-out check passed: the 14 matches at one-match grounds all get a neutral effect, so there is no leakage.
- Typical innings: 3.9% of the total after 1 over, 47.3% after 10 overs, 71.1% after 15, and 93.4% after 19. Scoring is back-loaded.
- Gap to par averages -0.3 in the powerplay, -0.1 in the middle and +2.9 at the death, with spreads of 8, 20 and 29 runs. The death-over bias is small (about a tenth of the spread) and does not affect the model.
- The gap to par after 14 overs correlates 0.89 with the final total's gap to a normal game. That is partly because the score so far is part of the final total, so it only confirms par is on the right scale.
- Outputs: venue_profile (95 grounds x 23 columns) and ball_master (200,961 balls x 100 columns).
- Reminder for notebook 04: the numeric ground effects and score_vs_par may be model inputs. pitch_type and spin_type tags may not.